# Part 1 — Notebook 02: Read LHE and Plot Generator-Level Kinematics

## Pedagogical Goal & Overview

In **Part 1 — Notebook 02**, you will read raw parton-level LHE event records, reconstruct $Z$ boson kinematics ($p_Z^\mu = p_b^\mu + p_{\bar{b}}^\mu$), compute physical observables ($p_T, \eta, \phi, \Delta R_{b\bar{b}}, m_{b\bar{b}}$), apply Monte Carlo weights, and plot 1D and 2D correlation histograms with Scikit-HEP `pylhe` and `mplhep`.

### Scientific Objectives
1. Parse LHE events non-destructively using `pylhe`.
2. Extract final-state partons ($b, \bar{b}, PID=\pm 5, status=1$) and event weights ($w_i$).
3. Reconstruct four-momentum vectors and compute kinematic observables using modular functions.
4. Compare $\Delta R_{b\bar{b}}$ against the high-boost analytical guide:
$$\Delta R_{b\bar{b}} \approx \frac{2m_Z}{p_T^Z}$$

## Step 1: Environment Setup & Strict Notebook 01 Output Check

We check whether the 1000-event production LHE file generated in Notebook 01 (`Zbbj_LO/Events/run_01/unweighted_events.lhe.gz`) exists in this Colab runtime.

> [!CAUTION]
> **No Remote Fallback**: If the output from Notebook 01 is missing, you must run `Part1_01_process_to_lhe.ipynb` first in your Google Colab runtime session to generate the LHE file.

In [ ]:
import os
import sys

print("Installing required dependencies (pylhe, mplhep, matplotlib, numpy)...")
!{sys.executable} -m pip install -q pylhe mplhep matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import pylhe

hep.style.use(hep.style.CMS)

# Strict check for Notebook 01 production output
lhe_candidates = [
    "Zbbj_LO/Events/run_01/unweighted_events.lhe.gz",
    "../Zbbj_LO/Events/run_01/unweighted_events.lhe.gz",
    "unweighted_events.lhe.gz"
]

target_lhe = None
for path in lhe_candidates:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        target_lhe = path
        break

if target_lhe is None:
    raise FileNotFoundError(
        "\n\n[ERROR]: Production LHE event file not found!\n"
        "Please execute 'Part1_01_process_to_lhe.ipynb' first in this Google Colab environment "
        "to generate the MadGraph LHE event file before running Notebook 02.\n"
    )

print(f"SUCCESS: Found LHE event file at: {target_lhe}")


> [!IMPORTANT]
> ### Self-Reflection & Coding Checkpoint 2.1
> 1. **Memory Management Question**: Why does `pylhe.read_lhe_with_attributes()` return a Python generator/iterator rather than loading all events into a list at once? What would happen if we loaded $10^7$ events into RAM?
> 2. **Numerical Trigonometry Question**: Why do we use `np.arctan2(py, px)` instead of `np.arctan(py / px)` when calculating azimuthal angle $\phi$?

<details>
<summary>Click to show Checkpoint 2.1 Reference Solution</summary>

<p>1. <b>Memory Streaming</b>: <code>pylhe.read_lhe_with_attributes()</code> yields events lazily one by one. Loading $10^7$ complex event objects into a list would consume gigabytes of RAM.</p>
<p>2. <b>Quadrant Safety</b>: <code>np.arctan2(py, px)</code> computes $\phi = \arctan(y/x)$ while preserving quadrant signs across $[-\pi, +\pi]$ and safely handling division by zero when $p_x = 0$.</p>

</details>

## Step 2: Extract Partons & Compute Kinematic Observables

Instead of executing a single monolithic loop, we modularize Step 2 into clean sub-steps with dedicated helper functions.

---

### Step 2a: Angular Kinematic Primitives ($\eta, \phi, \Delta\phi, \Delta R$)

In collider physics:
- **Pseudorapidity ($\eta$)**: $\eta = \frac{1}{2} \ln \left( \frac{p + p_z}{p - p_z} \right)$. Measures polar angle $\theta$ relative to the beam line ($\eta = 0$ is perpendicular to beam line).
- **Azimuthal Angle ($\phi$)**: Calculated using `np.arctan2(py, px)` to preserve quadrant signs in $[-\pi, +\pi]$.
- **Angular Distance ($\Delta R$)**: $\Delta R = \sqrt{(\Delta\eta)^2 + (\Delta\phi)^2}$, where $\Delta\phi \in [-\pi, +\pi]$ is wrapped across the $2\pi$ branch cut.

We define numerical helper functions with safety checks (preventing division by zero when $p = p_z$).

In [ ]:
def calculate_pseudorapidity(px, py, pz):
    # Calculate pseudorapidity eta = 0.5 * ln((p + pz) / (p - pz))
    p = np.sqrt(px**2 + py**2 + pz**2)
    # Prevent division by zero if particle is perfectly parallel to beam axis (p == pz)
    denom = max(1e-9, p - pz)
    return 0.5 * np.log((p + pz) / denom)


def calculate_azimuthal_angle(px, py):
    # Calculate azimuthal angle phi = arctan2(py, px) in [-pi, +pi]
    return np.arctan2(py, px)


def calculate_delta_phi(phi1, phi2):
    # Calculate folded azimuthal angle separation dphi in [-pi, +pi]
    dphi = phi1 - phi2
    return np.arctan2(np.sin(dphi), np.cos(dphi))


def calculate_delta_r(eta1, phi1, eta2, phi2):
    # Calculate angular separation Delta R = sqrt((d_eta)^2 + (d_phi)^2) in (eta, phi) space
    deta = eta1 - eta2
    dphi = calculate_delta_phi(phi1, phi2)
    return np.hypot(deta, dphi)

print("SUCCESS: Defined angular kinematic helper functions.")


### Step 2b: Four-Vector Addition & $Z$ Boson Reconstruction

By energy-momentum conservation, the four-momentum of the decayed $Z$ boson is the sum of its daughter partons:
$$p_Z^\mu = p_b^\mu + p_{\bar{b}}^\mu = (E_b + E_{\bar{b}}, \vec{p}_b + \vec{p}_{\bar{b}})$$

From $p_Z^\mu$, we compute:
1. **Transverse Momentum ($p_T^Z$)**: $p_T^Z = \sqrt{(p_x^Z)^2 + (p_y^Z)^2}$.
2. **Invariant Mass ($m_{b\bar{b}}$)**: $m_{b\bar{b}} = \sqrt{E_Z^2 - |\vec{p}_Z|^2}$, with $m^2 \ge 0$ safety against floating-point precision bounds.
3. **Decay Partons Separation ($\Delta R_{b\bar{b}}$)**: Using the angular helper functions defined in Step 2a.

In [ ]:
def reconstruct_z_boson_kinematics(b_part, bbar_part):
    # Extract 4-momentum components (Px, Py, Pz, E)
    px_b, py_b, pz_b, e_b = b_part.px, b_part.py, b_part.pz, b_part.e
    px_bbar, py_bbar, pz_bbar, e_bbar = bbar_part.px, bbar_part.py, bbar_part.pz, bbar_part.e

    # 1. Four-vector addition: p_Z^\mu = p_b^\mu + p_{bbar}^\mu
    px_z = px_b + px_bbar
    py_z = py_b + py_bbar
    pz_z = pz_b + pz_bbar
    e_z = e_b + e_bbar

    # 2. Transverse momentum pT = sqrt(Px^2 + Py^2)
    pt_z = np.hypot(px_z, py_z)
    
    # 3. Invariant mass m = sqrt(E^2 - |p|^2) with numerical safety
    p_sq_z = px_z**2 + py_z**2 + pz_z**2
    m_z = np.sqrt(max(0.0, e_z**2 - p_sq_z))

    # 4. Individual b and bbar kinematics
    pt_b = np.hypot(px_b, py_b)
    eta_b = calculate_pseudorapidity(px_b, py_b, pz_b)
    phi_b = calculate_azimuthal_angle(px_b, py_b)

    pt_bbar = np.hypot(px_bbar, py_bbar)
    eta_bbar = calculate_pseudorapidity(px_bbar, py_bbar, pz_bbar)
    phi_bbar = calculate_azimuthal_angle(px_bbar, py_bbar)

    # 5. Angular separation Delta R(b, bbar)
    delta_r = calculate_delta_r(eta_b, phi_b, eta_bbar, phi_bbar)

    return {
        "pt_z": pt_z,
        "m_z": m_z,
        "pt_b": pt_b,
        "pt_bbar": pt_bbar,
        "eta_b": eta_b,
        "eta_bbar": eta_bbar,
        "phi_b": phi_b,
        "phi_bbar": phi_bbar,
        "delta_r": delta_r
    }

print("SUCCESS: Defined Z boson reconstruction function.")


### Step 2c: LHE Generator Streaming & Event Array Extraction

We use `pylhe.read_lhe_with_attributes()` to stream LHE event records non-destructively:
- Filter final-state particles ($status = 1$) for bottom quark ($PID = 5$) and anti-bottom quark ($PID = -5$).
- Extract event weights ($w_i$).
- Aggregate kinematic values into NumPy arrays ready for plotting.

In [ ]:
def parse_and_process_lhe_file(lhe_filepath):
    z_pt_list, b_pt_list, bbar_pt_list = [], [], []
    b_eta_list, bbar_eta_list, delta_r_list = [], [], []
    m_bb_list, weights_list = [], []

    print(f"Reading and processing LHE events from: {lhe_filepath}")
    events = pylhe.read_lhe_with_attributes(lhe_filepath)

    for event in events:
        # Extract Monte Carlo event weight
        weight = getattr(event.eventinfo, 'weight', 1.0)
        b_part, bbar_part = None, None

        # Filter final-state particles (status == 1) for b (PID == 5) and bbar (PID == -5)
        for p in event.particles:
            if p.status == 1 and p.id == 5:
                b_part = p
            elif p.status == 1 and p.id == -5:
                bbar_part = p

        # If both decay partons are found, compute kinematics
        if b_part is not None and bbar_part is not None:
            kin = reconstruct_z_boson_kinematics(b_part, bbar_part)
            
            z_pt_list.append(kin["pt_z"])
            b_pt_list.append(kin["pt_b"])
            bbar_pt_list.append(kin["pt_bbar"])
            b_eta_list.append(kin["eta_b"])
            bbar_eta_list.append(kin["eta_bbar"])
            delta_r_list.append(kin["delta_r"])
            m_bb_list.append(kin["m_z"])
            weights_list.append(weight)

    return {
        "z_pt": np.array(z_pt_list),
        "b_pt": np.array(b_pt_list),
        "bbar_pt": np.array(bbar_pt_list),
        "b_eta": np.array(b_eta_list),
        "bbar_eta": np.array(bbar_eta_list),
        "delta_r": np.array(delta_r_list),
        "m_bb": np.array(m_bb_list),
        "weights": np.array(weights_list)
    }

print("SUCCESS: Defined LHE processing pipeline.")


### Step 2d: Execute Event Processing & Statistical Overview

We execute `parse_and_process_lhe_file(target_lhe)` on our production LHE file and inspect the weighted statistical means of our kinematic distributions.

In [ ]:
# Execute LHE event processing
data = parse_and_process_lhe_file(target_lhe)

# Unpack arrays for plotting
z_pt = data["z_pt"]
b_pt = data["b_pt"]
bbar_pt = data["bbar_pt"]
b_eta = data["b_eta"]
bbar_eta = data["bbar_eta"]
delta_r = data["delta_r"]
m_bb = data["m_bb"]
weights = data["weights"]

print(f"\n==========================================")
print(f"SUCCESS: Processed {len(z_pt)} events.")
print(f"Mean Z pT: {np.average(z_pt, weights=weights):.2f} GeV")
print(f"Mean m(bb): {np.average(m_bb, weights=weights):.2f} GeV")
print(f"Mean Delta R(b, bbar): {np.average(delta_r, weights=weights):.2f}")
print(f"==========================================")


> [!IMPORTANT]
> ### Self-Reflection & Physics Checkpoint 2.2
> 1. **Physics Question**: Why is the reconstructed invariant mass $m_{b\bar{b}}$ peaked at $91.2\text{ GeV}$? What physical effect causes events to scatter away from the exact peak?
> 2. **Coding Question**: How does `plt.hist(..., weights=weights)` handle Monte Carlo event weighting? Why would ignoring `weights` produce incorrect physics histograms for weighted MC samples?

<details>
<summary>Click to show Checkpoint 2.2 Reference Solution</summary>

<p>1. <b>Breit-Wigner Peak</b>: $m_{b\bar{b}}$ peaks at $m_Z = 91.1876\text{ GeV}$ following a Breit-Wigner distribution with natural decay width $\Gamma_Z \approx 2.495\text{ GeV}$.</p>
<p>2. <b>MC Weighting</b>: <code>plt.hist(..., weights=weights)</code> sums event weights per bin rather than counting events ($N_{\text{bin}} = \sum w_i$).</p>

</details>

## Step 3: Plot 1D Kinematic Histograms with `mplhep`

Now that we extracted kinematic observables into NumPy arrays, we visualize their distributions using `matplotlib` and `mplhep`.

---

### Step 3a: Example 1D Histogram — $Z$ Boson Transverse Momentum ($p_T^Z$)

Here is a fully working reference example demonstrating how to plot an MC-weighted 1D histogram using `plt.hist(..., weights=weights, histtype='step')`:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

# Plot Z pT histogram weighted by Monte Carlo weights
ax.hist(z_pt, bins=30, range=(100, 600), weights=weights, histtype='step', linewidth=2, color='navy', label='$Z$ boson')

ax.set_xlabel("Reconstructed $p_T^Z$ [GeV]", fontsize=12)
ax.set_ylabel("Weighted Events", fontsize=12)
ax.set_title("Transverse Momentum $p_T^Z$ Distribution", fontsize=13)
ax.axvline(150, color='red', linestyle='--', label='$p_T^Z > 150$ GeV Cut')
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Exercise 3b: Plot Bottom Quark Transverse Momentum ($p_T(b)$ & $p_T(\bar{b})$)
> Write code in the cell below to plot the transverse momentum distributions of the $b$ quark (`b_pt`, crimson line) and $\bar{b}$ quark (`bbar_pt`, dark orange dotted line) on the same plot across `range=(0, 400)` with `bins=30` and `weights=weights`.

<details>
<summary>Click to show Exercise 3b Reference Solution</summary>

<pre><code>fig, ax = plt.subplots(figsize=(7, 5))

ax.hist(b_pt, bins=30, range=(0, 400), weights=weights, histtype='step', linewidth=2, color='crimson', label='$b$ quark')
ax.hist(bbar_pt, bins=30, range=(0, 400), weights=weights, histtype='step', linewidth=2, color='darkorange', linestyle=':', label='$\bar{b}$ quark')

ax.set_xlabel("$p_T(b)$ [GeV]", fontsize=12)
ax.set_ylabel("Weighted Events", fontsize=12)
ax.set_title("Bottom Quark $p_T$ Distributions", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()</code></pre>

</details>

In [ ]:
# Exercise 3b Student Code Cell:
# TODO: Write code below to plot b_pt (crimson line) and bbar_pt (dark orange dotted line)
# Refer to the hidden reference solution block above if needed!

fig, ax = plt.subplots(figsize=(7, 5))

# --- YOUR CODE HERE ---


# ----------------------

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Exercise 3c: Plot Pseudorapidity Distributions ($\eta(b)$ & $\eta(\bar{b})$)
> Write code in the cell below to plot the pseudorapidity distributions of $\eta(b)$ (`b_eta`, purple line) and $\eta(\bar{b})$ (`bbar_eta`, teal dotted line) across `range=(-4, 4)` with `bins=30` and `weights=weights`.

<details>
<summary>Click to show Exercise 3c Reference Solution</summary>

<pre><code>fig, ax = plt.subplots(figsize=(7, 5))

ax.hist(b_eta, bins=30, range=(-4, 4), weights=weights, histtype='step', linewidth=2, color='purple', label='$\eta(b)$')
ax.hist(bbar_eta, bins=30, range=(-4, 4), weights=weights, histtype='step', linewidth=2, color='teal', linestyle=':', label='$\eta(\bar{b})$')

ax.set_xlabel("Pseudorapidity $\eta$", fontsize=12)
ax.set_ylabel("Weighted Events", fontsize=12)
ax.set_title("Bottom Quark Pseudorapidity $\eta$ Distributions", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()</code></pre>

</details>

In [ ]:
# Exercise 3c Student Code Cell:
# TODO: Write code below to plot b_eta (purple line) and bbar_eta (teal dotted line)
# Refer to the hidden reference solution block above if needed!

fig, ax = plt.subplots(figsize=(7, 5))

# --- YOUR CODE HERE ---


# ----------------------

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Exercise 3d: Plot Reconstructed Invariant Mass ($m_{b\bar{b}}$)
> Write code in the cell below to plot the reconstructed invariant mass `m_bb` across `range=(60, 120)` with `bins=30`, dark green line, and overlay a vertical reference line at the PDG $Z$ boson mass ($m_Z = 91.2\text{ GeV}$).

<details>
<summary>Click to show Exercise 3d Reference Solution</summary>

<pre><code>fig, ax = plt.subplots(figsize=(7, 5))

ax.hist(m_bb, bins=30, range=(60, 120), weights=weights, histtype='step', linewidth=2, color='darkgreen', label='$m_{b\bar{b}}$')
ax.axvline(91.1876, color='black', linestyle='--', label='PDG $m_Z = 91.2$ GeV')

ax.set_xlabel("Invariant Mass $m_{b\bar{b}}$ [GeV]", fontsize=12)
ax.set_ylabel("Weighted Events", fontsize=12)
ax.set_title("Reconstructed $Z$ Boson Invariant Mass", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()</code></pre>

</details>

In [ ]:
# Exercise 3d Student Code Cell:
# TODO: Write code below to plot m_bb with PDG m_Z line (91.2 GeV)
# Refer to the hidden reference solution block above if needed!

fig, ax = plt.subplots(figsize=(7, 5))

# --- YOUR CODE HERE ---


# ----------------------

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Exercise 3e: Plot Angular Separation ($\Delta R_{b\bar{b}}$)
> Write code in the cell below to plot the angular separation `delta_r` between bottom decay partons across `range=(0, 3.5)` with `bins=30` and `weights=weights`.

<details>
<summary>Click to show Exercise 3e Reference Solution</summary>

<pre><code>fig, ax = plt.subplots(figsize=(7, 5))

ax.hist(delta_r, bins=30, range=(0, 3.5), weights=weights, histtype='step', linewidth=2, color='chocolate', label='$\Delta R_{b\bar{b}}$')

ax.set_xlabel("Angular Separation $\Delta R_{b\bar{b}}$", fontsize=12)
ax.set_ylabel("Weighted Events", fontsize=12)
ax.set_title("Parton Opening Angle $\Delta R_{b\bar{b}}$ Distribution", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()</code></pre>

</details>

In [ ]:
# Exercise 3e Student Code Cell:
# TODO: Write code below to plot delta_r
# Refer to the hidden reference solution block above if needed!

fig, ax = plt.subplots(figsize=(7, 5))

# --- YOUR CODE HERE ---


# ----------------------

plt.tight_layout()
plt.show()


## Step 4: 2D Correlation $\Delta R_{b\bar{b}}$ vs. $p_T^Z$ & Boost Guide Overlay

As $p_T^Z$ increases, relativistic boost collimates the decay products ($b, \bar{b}$). We overlay the analytical high-boost two-body guide line:
$$\Delta R_{b\bar{b}} = \frac{2m_Z}{p_T^Z}$$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

h = ax.hist2d(z_pt, delta_r, bins=[30, 30], range=[[100, 600], [0, 3.5]], weights=weights, cmap='viridis')
plt.colorbar(h[3], ax=ax, label='Weighted Events')

pt_grid = np.linspace(120, 600, 200)
m_z_pdg = 91.1876
delta_r_guide = (2.0 * m_z_pdg) / pt_grid

ax.plot(pt_grid, delta_r_guide, color='red', linestyle='--', linewidth=2.5, label='High-Boost Guide: $\\Delta R = \\frac{2 m_Z}{p_T^Z}$')

ax.set_xlabel("Reconstructed $p_T^Z$ [GeV]", fontsize=13)
ax.set_ylabel("Angular Separation $\\Delta R_{b\\bar{b}}$", fontsize=14)
ax.set_title("2D Correlation: $\\Delta R_{b\\bar{b}}$ vs. $p_T^Z$ in $pp \\to Z+j, Z \\to b\\bar{b}$", fontsize=14)
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


> [!IMPORTANT]
> ### Section Checkpoint & Quantitative Exercise 2.3
> 1. **Collimation Calculation**: Compute the approximate opening angle $\Delta R \approx \frac{2m_Z}{p_T^Z}$ for $p_T^Z = 250\text{ GeV}$, $400\text{ GeV}$, and $600\text{ GeV}$.
> 2. **Large-$R$ Jet Threshold**: Standard small-$R$ jets use $R=0.4$, while large-$R$ jets use $R=0.8$ or $R=1.0$. At what $p_T^Z$ threshold will both $b$ quarks begin to fall inside a single $R=0.8$ large-$R$ jet?

<details>
<summary>Click to show Quantitative Exercise 2.3 Reference Solution</summary>

<p>1. <b>Opening Angles</b>:<br>
- At $p_T^Z = 250\text{ GeV}$: $\Delta R \approx \frac{2 \times 91.2}{250} \approx 0.73$<br>
- At $p_T^Z = 400\text{ GeV}$: $\Delta R \approx \frac{2 \times 91.2}{400} \approx 0.46$<br>
- At $p_T^Z = 600\text{ GeV}$: $\Delta R \approx \frac{2 \times 91.2}{600} \approx 0.30$</p>
<p>2. <b>Single Large-$R$ Jet Containment</b>: For both decay partons to fall inside a single $R=0.8$ jet, $\Delta R \le 0.8 \implies p_T^Z \ge \frac{2 \times 91.2}{0.8} \approx 228\text{ GeV}$. For $p_T^Z \ge 200\text{ GeV}$, the $Z \to b\bar{b}$ system forms a single merged large-$R$ jet!</p>

</details>